# PLM ADA architecture audit on synthetic tensors

This notebook audits three implementation properties from `PLM_ADA/PLM.py` using abstract numeric fixtures. It tests whether sequence-wide head selection is prefix invariant, whether selecting experts reduces calls in the current evaluation order, and whether the pairwise-distance primitive has basic mathematical invariants.

**Run locally:** Python 3.11+, PyTorch 2.6+, Jupyter, then **Restart Kernel and Run All Cells**. All values are small generated numeric tensors. This notebook contains no biological sequences, biological feature encoders, sequence generation, sequence prediction training, dataset or checkpoint download.

### Source scope

`DynamicSparseGeometricGQA.forward` derives a sequence-wide head mask with mean pooling before output masking. `FineGrainedMoE.forward` computes every expert output before a weighted combination. `torch.cdist` forms the distance feature in the geometric branch. The tests below isolate those operators. They do not reproduce the full model, its configured heads, or its biological task.

In [1]:
import math
import platform
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 27
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print({"python": platform.python_version(), "torch": torch.__version__, "device": "cpu", "seed": SEED})

{'python': '3.11.0rc2', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 27}


### Verify the effect of future inputs on the head mask

Use two fixed numeric sequences with the same prefix and different suffixes. The sigmoid and top-k calculation reproduces the head-selection order from the source. In a causal prefix test, a change in a later position should not change the selected heads for an unchanged earlier position. The fixed prefix control is an invariant check for the test fixture.

In [2]:
prefix = torch.tensor([[[2.0, -2.0], [2.0, -2.0]]])
seq_a = torch.cat((prefix, torch.tensor([[[6.0, -6.0], [6.0, -6.0]]])), dim=1)
seq_b = torch.cat((prefix, torch.tensor([[[-6.0, 6.0], [-6.0, 6.0]]])), dim=1)
probe = torch.eye(2)

def select_heads(values):
    scores = torch.sigmoid(values.mean(dim=1) @ probe)
    best, positions = scores.topk(1, dim=-1)
    return torch.zeros_like(scores).scatter(1, positions, best)

assert torch.equal(seq_a[:, :2], seq_b[:, :2])
a, b = select_heads(seq_a), select_heads(seq_b)
causal_control = torch.equal(select_heads(seq_a[:, :2]), select_heads(seq_b[:, :2]))
assert causal_control
head_gate_prefix_invariant = torch.equal(a, b)
print({"prefix_control": causal_control, "head_gate_prefix_invariant": head_gate_prefix_invariant,
       "head_mask_a": a.tolist(), "head_mask_b": b.tolist()})

{'prefix_control': True, 'head_gate_prefix_invariant': False, 'head_mask_a': [[0.9820137619972229, 0.0]], 'head_mask_b': [[0.0, 0.8807970285415649]]}


### Count expert calls separately from selected weights

The source combines only selected outputs, but constructs the full `[token, expert, feature]` tensor first. Here each expert is a fixed arithmetic operator. A call counter records the work per token and a gradient check verifies that the abstract weighted-combination computation is differentiable.

In [3]:
x = torch.arange(12, dtype=torch.float32).reshape(4, 3).div(10).requires_grad_()
logits = torch.tensor([[4.0, 0.0, 1.0], [4.0, 1.0, 0.0],
                       [0.0, 4.0, 1.0], [1.0, 4.0, 0.0]])
top_idx = logits.topk(1, dim=1).indices
weights = torch.softmax(logits.gather(1, top_idx), dim=-1)
counts = [0, 0, 0]

def fixed_expert(j, values):
    counts[j] += len(values)
    return values * float(j + 1)

dense_outputs = torch.stack([fixed_expert(j, x) for j in range(3)], dim=1)
combine = torch.zeros_like(logits).scatter(1, top_idx, weights)
result = torch.einsum("te,ted->td", combine, dense_outputs)
expected = x * (top_idx + 1)
torch.testing.assert_close(result, expected)
result.square().mean().backward()
assert x.grad is not None and torch.isfinite(x.grad).all().item()
print({"computed_token_expert_pairs": sum(counts),
       "selected_token_expert_pairs": top_idx.numel(),
       "pairs_per_expert": counts, "combination_gradient_finite": True})

{'computed_token_expert_pairs': 12, 'selected_token_expert_pairs': 4, 'pairs_per_expert': [4, 4, 4], 'combination_gradient_finite': True}


### Verify the generic distance primitive

Use a three-point abstract coordinate system. Check shape, symmetry, zero diagonal, a known 3–4–5 distance, and translation invariance. These checks concern the numeric primitive only. No structural prediction or biological utility follows from them.

In [4]:
points = torch.tensor([[[0.0, 0.0, 0.0], [3.0, 0.0, 0.0], [0.0, 4.0, 0.0]]])
distances = torch.cdist(points, points, p=2)
expected = torch.tensor([[[0.0, 3.0, 4.0], [3.0, 0.0, 5.0], [4.0, 5.0, 0.0]]])
torch.testing.assert_close(distances, expected)
torch.testing.assert_close(distances, distances.transpose(-1, -2))
translated = points + torch.tensor([6.0, -2.0, 1.0])
torch.testing.assert_close(distances, torch.cdist(translated, translated, p=2))
print({"distance_check": "passed",
       "head_gate_prefix_invariance": "pass" if head_gate_prefix_invariant else "fail",
       "sparse_dispatch_evidenced": sum(counts) == top_idx.numel()})

{'distance_check': 'passed', 'head_gate_prefix_invariance': 'fail', 'sparse_dispatch_evidenced': False}


### Interpretation

The expected evidence is an exact distance check, a failing prefix gate for sequence-wide head masking, and more computed than selected expert pairs. These are limited implementation observations on fixed arrays. Treat source names that suggest sparse execution or architectural parity as hypotheses requiring independent evidence.